In [ ]:
import sys
from collections import defaultdict
import threading

class DB:
    
    def __init__(self):
        self.data = {}
        self.counts = defaultdict(int)
        self.lock = threading.Lock()
        self.idx = 0
        self.txns = defaultdict(list)
    
    def set(self, key, value):
        with self.lock:
            if self.idx:
                old = self.data.get(key)
                self.txns[self.idx].append((key,old))
            self._set(key, value)
    
    def _set(self, key, value):
        old = self.data.get(key)
        if old is not None:
            self.counts[old] -= 1
        self.data[key] = value
        self.counts[value] += 1
        
    def get(self, key):
        return self.data.get(key, "NULL")
    
    def unset(self, key):
        if self.idx:
            old = self.data.get(key)
            self.txns[self.idx].append((key,old))
        self._unset(key)
    
    def _unset(self, key):
        old = self.data.get(key)
        if old is None:
            return
        self.counts[old] -= 1
        self.data.pop(key, None)

    def num_equal_to(self, value):
        return self.counts.get(value, 0)
    
    def begin(self):
        self.idx += 1
        
    def rollback(self):
        if self.idx == 0:
            return "NO TRANSACTION"
        txn = self.txns.pop(self.idx)
        self.idx -= 1
        while txn:
            key, old = txn.pop()
            if old is None:
                self._unset(key)
            else:
                self._set(key, old)
        return None
        
    def commit(self):
        if self.idx == 0:
            return "NO TRANSACTION"
        self.idx = 0
        self.txns.clear()
        return None
        
if __name__ == "__main__":
    db = DB()
    for line in sys.stdin:
        words = line.split()
        if not words:
            continue
        if words[0] == "SET":
            db.set(words[1],words[2])

In [4]:
from collections import defaultdict
tab = defaultdict(int)
tab['a'] +=1

In [6]:
tab['b']

0

In [7]:
tab

defaultdict(int, {'a': 1, 'b': 0})